# Практическая работа 4. Современные симметричные шифры  
## Шифр «Магма»

Характеристики:  
  Длина блока:  64 бита (8 байт)  
  Длина ключа:  256 бит (32 байта)  
  Число раундов: 32

In [1]:
'''импорт библиотек'''
import sys
import os

In [2]:
'''Значения подстановок π', определены в ГОСТ Р 34.12-2015 (раздел 5.1)и 
представляют собой фиксированные таблицы нелинейного биективного преобразования, 
используемые далее в коде, в блоке замен шифра «Магма».'''
#  π0' = (12, 4, 6, 2, 10, 5, 11, 9, 14, 8, 13, 7, 0, 3, 15, 1);
#  π1' = (6, 8, 2, 3, 9, 10, 5, 12, 1, 14, 4, 7, 11, 13, 0, 15);
#  π2' = (11, 3, 5, 8, 2, 15, 10, 13, 14, 1, 7, 4, 12, 9, 6, 0);
#  π3' = (12, 8, 2, 1, 13, 4, 15, 6, 7, 0, 10, 5, 3, 14, 9, 11);
#  π4' = (7, 15, 5, 10, 8, 1, 6, 13, 0, 9, 3, 14, 11, 4, 2, 12);
#  π5' = (5, 13, 15, 6, 9, 2, 12, 10, 11, 7, 8, 1, 4, 3, 14, 0);
#  π6' = (8, 14, 2, 5, 6, 9, 1, 12, 15, 4, 11, 0, 13, 10, 3, 7);
#  π7' = (1, 7, 14, 13, 0, 5, 8, 3, 4, 15, 10, 6, 9, 12, 11, 2).
# ================================================================
K = [
    [12, 4, 6, 2, 10, 5, 11, 9, 14, 8, 13, 7, 0, 3, 15, 1],
    [6, 8, 2, 3, 9, 10, 5, 12, 1, 14, 4, 7, 11, 13, 0, 15],
    [11, 3, 5, 8, 2, 15, 10, 13, 14, 1, 7, 4, 12, 9, 6, 0],
    [12, 8, 2, 1, 13, 4, 15, 6, 7, 0, 10, 5, 3, 14, 9, 11],
    [7, 15, 5, 10, 8, 1, 6, 13, 0, 9, 3, 14, 11, 4, 2, 12],
    [5, 13, 15, 6, 9, 2, 12, 10, 11, 7, 8, 1, 4, 3, 14, 0],
    [8, 14, 2, 5, 6, 9, 1, 12, 15, 4, 11, 0, 13, 10, 3, 7],
    [1, 7, 14, 13, 0, 5, 8, 3, 4, 15, 10, 6, 9, 12, 11, 2],
]

In [3]:
'''КЗУ'''
#  256-битный ключ делится на 8 частей по 32 бита
#  Порядок использования подключей в раундах (раздел 5.3 ГОСТ Р 34.12-2015):
ENCRYPT_KEYS = [0,1,2,3,4,5,6,7] * 3 + [7,6,5,4,3,2,1,0]
DECRYPT_KEYS = [0,1,2,3,4,5,6,7] + [7,6,5,4,3,2,1,0] * 3

In [4]:
"""Функция для разбивания 256-битного ключа на 8 подключей"""
def split_key(key_bytes: bytes) -> list:
    return [int.from_bytes(key_bytes[i*4:(i+1)*4], 'big') for i in range(8)]

In [5]:
"""Нелинейная функция Ψ"""
# Ψ(𝑇,𝑋) = 𝑅(𝐾(𝑇⊞𝑋))
def psi(t: int, x: int) -> int:
    s = (t + x) & 0xFFFFFFFF                                 # (T+X)mod2³²
    s = sum(K[i][(s >> 4*i) & 0xF] << 4*i for i in range(8)) # K
    return ((s << 11) | (s >> 21)) & 0xFFFFFFFF              # R

In [6]:
'''Зашифрование и расшифрование одного блока'''
# ГОСТ Р 34.12-2015:
# G[k](a1, a0) = (a0, g[k](a0) ⊕ a1) раунды 1–31 (ф 16)
# G*[k](a1, a0) = (g[k](a0) ⊕ a1)||a0 раунд 32 (ф 17)  
# E(a) = G*[K32]G[K31]...G[K2]G[K1](a1, a0) Зашифрование (ф 19)
# D(a) = G*[K1]G[K2]...G[K31]G[K32](a1, a0) Расшифрование (ф 20)

"""Зашифрование"""
def encrypt_block(block: bytes, subkeys: list) -> bytes:
    a1 = int.from_bytes(block[0:4], 'big')  #левая половина
    a0 = int.from_bytes(block[4:8], 'big')  #правая половина
    for i in range(31):                                      #раунды 1–31
        a1, a0 = a0, a1 ^ psi(a0, subkeys[ENCRYPT_KEYS[i]])
    a1 = a1 ^ psi(a0, subkeys[ENCRYPT_KEYS[31]])             #раунд 32
    return a1.to_bytes(4, 'big') + a0.to_bytes(4, 'big')

"""Расшифрование"""
def decrypt_block(block: bytes, subkeys: list) -> bytes:
    a1 = int.from_bytes(block[0:4], 'big')
    a0 = int.from_bytes(block[4:8], 'big')
    for i in range(31):
        a1, a0 = a0, a1 ^ psi(a0, subkeys[DECRYPT_KEYS[i]])
    a1 = a1 ^ psi(a0, subkeys[DECRYPT_KEYS[31]])
    return a1.to_bytes(4, 'big') + a0.to_bytes(4, 'big')

In [7]:
'''Дополнение данных до размера блока'''
def pad(data: bytes) -> bytes:
    n = 8 - (len(data) % 8)
    return data + bytes([n] * n)

def unpad(data: bytes) -> bytes:
    if not data:
        raise ValueError("Нет данных")
    n = data[-1]
    if n < 1 or n > 8 or data[-n:] != bytes([n] * n):
        raise ValueError("Некорректно")
    return data[:-n]

In [8]:
'''Режим простой замены (ECB)'''
# Каждый блок зашифровывается независимо от остальных.

def encrypt_ecb(data: bytes, subkeys: list) -> bytes:
    data = pad(data)
    result = bytearray()
    for i in range(0, len(data), 8):
        result += encrypt_block(data[i:i+8], subkeys)
    return bytes(result)


def decrypt_ecb(data: bytes, subkeys: list) -> bytes:
    if len(data) % 8 != 0:
        raise ValueError("должно быть кратна 8 байтам")
    result = bytearray()
    for i in range(0, len(data), 8):
        result += decrypt_block(data[i:i+8], subkeys)
    return unpad(bytes(result))

In [19]:
'''Тестирование по ГОСТ Р 34.12-2015 (Приложение А.2)'''

def run_test():
    key_hex = "ffeeddccbbaa99887766554433221100f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff"
    a  = "fedcba9876543210"
    b  = "4ee901e5c2d8ca3d"

    key = bytes.fromhex(key_hex)
    pt  = bytes.fromhex(a)
    ct_expected = bytes.fromhex(b)
    subkeys = split_key(key)

    print("  Тест по ГОСТ Р 34.12-2015 (Приложение А.2)")
    print(f"  Ключ: {key_hex}")
    print(f"  Открытый текст: {a}")
    print(f"  Ожидаемый шифртекст: {b}")
    print()

    ct = encrypt_block(pt, subkeys)
    print(f"  Полученный шифртекст: {ct.hex()}")
    assert ct == ct_expected, "ОШИБКА зашифрования!"
    print("  Зашифрование ВЕРНО!!!")

    dt = decrypt_block(ct, subkeys)
    print(f"  Расшифрованный текст: {dt.hex()}")
    assert dt == pt, "ОШИБКА расшифрования!"
    print("  Расшифрование ВЕРНО!!!")

    return True

In [20]:
run_test()

  Тест по ГОСТ Р 34.12-2015 (Приложение А.2)
  Ключ: ffeeddccbbaa99887766554433221100f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff
  Открытый текст: fedcba9876543210
  Ожидаемый шифртекст: 4ee901e5c2d8ca3d

  Полученный шифртекст: 4ee901e5c2d8ca3d
  Зашифрование ВЕРНО!!!
  Расшифрованный текст: fedcba9876543210
  Расшифрование ВЕРНО!!!


True

In [24]:
def main():
    print("Шифр «Магма»")

    input_path = input("1. Путь к входному файлу: ")
    with open(input_path, 'rb') as f:
        data = f.read()

    key_hex = input("2. Ключ (64 символа): ")
    subkeys = split_key(bytes.fromhex(key_hex))

    action = input("3. Действие (e — зашифровать / d — расшифровать): ")

    output_path = input("4. Путь к выходному файлу: ")

    if action == 'e':
        result = encrypt_ecb(data, subkeys)
    else:
        result = decrypt_ecb(data, subkeys)

    with open(output_path, 'wb') as f:
        f.write(result)

In [22]:
if __name__ == '__main__':
    main()

  Шифр «Магма» (ГОСТ Р 34.12-2015)


1) Путь к входному файлу:  /home/miha/Рабочий стол/test.txt
2) Ключ (64 hex-символа):  ffeeddccbbaa99887766554433221100f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff
3) Действие (e — зашифровать / d — расшифровать):  e
4) Путь к выходному файлу:  out.txt


In [23]:
if __name__ == '__main__':
    main()

  Шифр «Магма» (ГОСТ Р 34.12-2015)


1) Путь к входному файлу:  /home/miha/Рабочий стол/out.txt
2) Ключ (64 hex-символа):  ffeeddccbbaa99887766554433221100f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff
3) Действие (e — зашифровать / d — расшифровать):  d
4) Путь к выходному файлу:  in.txt
